In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df = pd.read_csv('/workspaces/customer_chrun_prediction/data/processed/feature_engineered_customer_churn_dataset.csv')
df.head(2)

,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn,...,monthly_charges_log,high_price_flag,price_to_tenure_ratio,is_auto_pay,high_support_calls,support_calls_per_month,recent_issue_proxxy,high_price_new_customer,month_to_month_high_support,long_term_low_support
0,52,54.20,2818.4,Month-to-month,Credit,DSL,No,Yes,1,No,...,4.010963,0,1.042308,1,0,0.019231,0,0,0,1
1,15,35.28,529.2,Month-to-month,Debit,DSL,No,No,2,No,...,3.591267,0,2.352000,1,0,0.133333,0,0,0,0


In [5]:
# Binary categorical columns
binary_cols = ['tech_support','online_security','churn']
# Map Yes/No and Male/Female to 0/1
df[binary_cols] = df[binary_cols].replace({
    'Yes': 1, 'No': 0})

/tmp/ipykernel_21007/2553764290.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_cols] = df[binary_cols].replace({


In [6]:
df.head(2)

,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn,...,monthly_charges_log,high_price_flag,price_to_tenure_ratio,is_auto_pay,high_support_calls,support_calls_per_month,recent_issue_proxxy,high_price_new_customer,month_to_month_high_support,long_term_low_support
0,52,54.20,2818.4,Month-to-month,Credit,DSL,0,1,1,0,...,4.010963,0,1.042308,1,0,0.019231,0,0,0,1
1,15,35.28,529.2,Month-to-month,Debit,DSL,0,0,2,0,...,3.591267,0,2.352000,1,0,0.133333,0,0,0,0


In [7]:
# one hot encoding 
multi_cat_cols = ['contract','payment_method','internet_service']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
df.head(2)

,tenure,monthly_charges,total_charges,tech_support,online_security,support_calls,churn,is_new_customer,is_long_term_customer,monthly_charges_log,...,recent_issue_proxxy,high_price_new_customer,month_to_month_high_support,long_term_low_support,contract_One year,contract_Two year,payment_method_Credit,payment_method_Debit,payment_method_UPI,internet_service_Fiber
0,52,54.20,2818.4,0,1,1,0,0,1,4.010963,...,0,0,0,1,False,False,True,False,False,False
1,15,35.28,529.2,0,0,2,0,0,0,3.591267,...,0,0,0,0,False,False,False,True,False,False


Machine learning experiment

In [8]:
df['churn'].value_counts()

churn
0    13157
1     6843
Name: count, dtype: int64

In [9]:
# prepare data for modeling
X = df.drop('churn', axis=1)
y = df['churn']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [10]:
# Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=300,
                                class_weight='balanced',
                                  random_state=42,
                                  n_jobs=-1)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.88      2631
           1       0.84      0.65      0.73      1369

    accuracy                           0.84      4000
   macro avg       0.84      0.79      0.81      4000
weighted avg       0.84      0.84      0.83      4000

Confusion Matrix:
[[2465  166]
 [ 485  884]]


In [11]:
# lightGBM Classifier
import time


lgbm = lgb.LGBMClassifier(n_estimators=300,
                            learning_rate=0.05,
                            class_weight='balanced',
                            random_state=42,
                            n_jobs=-1)
# training timer
strat_train =time.time()
lgbm.fit(X_train,y_train)
train_time = time.time() - strat_train
print(f"LightGBM training time: {train_time:.2f} seconds")

# predictions
y_pred = lgbm.predict(X_test)
# Classification report
print(classification_report(y_test, y_pred, digits=2))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))




[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 5474, number of negative: 10526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1349
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM training time: 0.76 seconds
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      2631
           1       0.84      0.66      0.74      1369

    accuracy                           0.84      4000
   macro avg       0.84      0.80      0.81      4000
weighted avg       0.84      0.84      0.84      4000


In [13]:
# xgb
xgb = XGBClassifier(n_estimators=300,
                      learning_rate=0.05,
                      scale_pos_weight= (y_train == 0).sum() / (y_train == 1).sum(),
                      random_state=42,
                      n_jobs=-1)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print("XGBoost Classifier Report:")
print(classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))




XGBoost Classifier Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      2631
           1       0.85      0.66      0.74      1369

    accuracy                           0.84      4000
   macro avg       0.85      0.80      0.82      4000
weighted avg       0.84      0.84      0.84      4000

Confusion Matrix:
[[2473  158]
 [ 467  902]]


In [15]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

# Objective function for Optuna
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss"
    }
    
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = xgb.predict(X_test) # Keep your tuned threshold
    return recall_score(y_test, y_pred, pos_label=1)  # Optimize recall for churners

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Recall:", study.best_value)

[I 2026-01-09 03:19:57,417] A new study created in memory with name: no-name-1e2bc411-a807-421d-bee1-cab491c2ce78
[I 2026-01-09 03:20:05,075] Trial 0 finished with value: 0.6588750913075238 and parameters: {'n_estimators': 561, 'learning_rate': 0.16932321223330718, 'max_depth': 6, 'subsample': 0.6603284477323712, 'colsample_bytree': 0.5566305216094041, 'min_child_weight': 8, 'gamma': 4.7434797553940875, 'reg_alpha': 3.2915014469917816, 'reg_lambda': 1.0758424867085865}. Best is trial 0 with value: 0.6588750913075238.
[I 2026-01-09 03:20:14,086] Trial 1 finished with value: 0.6588750913075238 and parameters: {'n_estimators': 637, 'learning_rate': 0.017379472084016256, 'max_depth': 4, 'subsample': 0.565022219816435, 'colsample_bytree': 0.7988482013593856, 'min_child_weight': 9, 'gamma': 0.25935396859947113, 'reg_alpha': 1.357754857356257, 'reg_lambda': 4.999484905111978}. Best is trial 0 with value: 0.6588750913075238.
[I 2026-01-09 03:20:22,010] Trial 2 finished with value: 0.6588750913

Best Params: {'n_estimators': 561, 'learning_rate': 0.16932321223330718, 'max_depth': 6, 'subsample': 0.6603284477323712, 'colsample_bytree': 0.5566305216094041, 'min_child_weight': 8, 'gamma': 4.7434797553940875, 'reg_alpha': 3.2915014469917816, 'reg_lambda': 1.0758424867085865}
Best Recall: 0.6588750913075238


In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import time

# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Add the scale_pos_weight and fixed params to the best ones from Optuna
best_params = study.best_params
best_params.update({
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "logloss"
})

# Create model from best params
xgb = XGBClassifier(**best_params)

# Training timer
start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"⏱ Training time: {train_time:.2f} seconds")

# Prediction timer
start_pred = time.time()
proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.5).astype(int)  # Keep your tuned threshold
pred_time = time.time() - start_pred
print(f"⏱ Prediction time: {pred_time:.4f} seconds")

# Classification report
print(classification_report(y_test, y_pred, digits=3))


⏱ Training time: 7.63 seconds
⏱ Prediction time: 0.0377 seconds
              precision    recall  f1-score   support

           0      0.840     0.909     0.873      2631
           1      0.793     0.668     0.725      1369

    accuracy                          0.827      4000
   macro avg      0.817     0.789     0.799      4000
weighted avg      0.824     0.827     0.823      4000



In [19]:
import mlflow
import mlflow.sklearn  # or mlflow.xgboost
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, roc_auc_score
import time
import os

# Force MLflow to always use the project root's mlruns folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
mlflow.set_tracking_uri(f"file://{project_root}/mlruns")
mlflow.set_experiment("Telco Churn - XGBoost")

with mlflow.start_run():
    # Calculate scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    # Best params from Optuna
    best_params = study.best_params
    best_params.update({
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "logloss"
    })

    # Log parameters
    mlflow.log_params(best_params)

    # Training timer
    start_train = time.time()
    xgb = XGBClassifier(**best_params)
    xgb.fit(X_train, y_train)
    train_time = time.time() - start_train
    mlflow.log_metric("train_time", train_time)

    # Prediction
    start_pred = time.time()
    proba = xgb.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.5).astype(int)  # Keep your tuned threshold
    pred_time = time.time() - start_pred
    mlflow.log_metric("pred_time", pred_time)

    # Metrics
    precision = precision_score(y_test, y_pred, pos_label=1)
    recall = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    auc = roc_auc_score(y_test, proba)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)

    # Save model
    mlflow.xgboost.log_model(xgb, "model")

    print(classification_report(y_test, y_pred, digits=3))


2026/01/09 03:26:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


              precision    recall  f1-score   support

           0      0.840     0.909     0.873      2631
           1      0.793     0.668     0.725      1369

    accuracy                          0.827      4000
   macro avg      0.817     0.789     0.799      4000
weighted avg      0.824     0.827     0.823      4000

